# ResNet Multi-Class Classification

## Importing Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import json
import joblib
import timm
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import CIFAR100
from torchvision.utils import make_grid
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import random_split, ConcatDataset
from torchvision import transforms
from loguru import logger
import matplotlib.image as mpimg
from sklearn.preprocessing import LabelEncoder
from nazi_symbols_classification.training.data_preparation import get_image_paths, load_labels_df
from nazi_symbols_classification.training.torch_dataset_preparation import ImageData, get_device, to_device, ToDeviceLoader
from early_stopping_pytorch import EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


## Loading Data

In [2]:
dir_name = os.path.dirname(os.getcwd())
dataset_path = f"{dir_name}/datasets/nazi-symbols-classification"

In [4]:
train_labels_df = load_labels_df(dataset_path, "train", binary=False)
valid_labels_df = load_labels_df(dataset_path, "val", binary=False)
test_labels_df = load_labels_df(dataset_path, "test", binary=False)

2025-06-10 16:23:02.054 | INFO     | __main__:load_labels_df:3 - Number of images in train is 9009
2025-06-10 16:23:02.064 | INFO     | __main__:load_labels_df:3 - Number of images in val is 1934
2025-06-10 16:23:02.070 | INFO     | __main__:load_labels_df:3 - Number of images in test is 1944


## Data Preprocessing

In [5]:
remap_dict = {
    'nazi_cls': {
        "atomwaffen": "neo-nazi", 
        "blood_honor_emblem": "neo-nazi", 
        "celtic_cross": "neo-nazi", 
        "combat_18_emblem": "neo-nazi", 
        "golden_dawn": "neo-nazi", 
        "hammerskins": "neo-nazi", 
        "identitaere_bewegung_emblem": "neo-nazi", 
        "kolovrat": "neo-nazi", 
        "national_rebirth_poland": "neo-nazi", 
        "volksfront_emblem": "neo-nazi", 
        "doppelsiegrune": "siegrune"
    }
}
train_labels_df = train_labels_df.replace(remap_dict)
valid_labels_df = valid_labels_df.replace(remap_dict)
test_labels_df = test_labels_df.replace(remap_dict)

In [6]:
le = LabelEncoder()

le.fit(train_labels_df.nazi_cls)
train_labels_df["nazi_cls_encoded"] = le.transform(train_labels_df.nazi_cls)
valid_labels_df["nazi_cls_encoded"] = le.transform(valid_labels_df.nazi_cls)
test_labels_df["nazi_cls_encoded"] = le.transform(test_labels_df.nazi_cls)

## Data Transformation

In [8]:
stats = ((0.5074,0.4867,0.4411),(0.2011,0.1987,0.2025))
data_transf = transforms.Compose([transforms.ToPILImage(), 
                                  transforms.Grayscale(num_output_channels=3),
                                  transforms.Resize((224, 224)), 
                                  transforms.ToTensor(),
                                  transforms.Normalize(*stats)])

def create_datasets(batch_size, dataset_path=dataset_path):
    train_data = ImageData(df = train_labels_df,
                           data_directory = os.path.join(dataset_path, 'train'),
                           transform = data_transf,
                           label_column = "nazi_cls_encoded")
    train_loader = DataLoader(dataset = train_data, batch_size = batch_size, shuffle=True)
    
    valid_data = ImageData(df = valid_labels_df, 
                           data_directory = os.path.join(dataset_path, 'val'), 
                           transform = data_transf,
                           label_column = "nazi_cls_encoded")
    valid_loader = DataLoader(dataset = valid_data, batch_size = batch_size, shuffle=False)

    test_data = ImageData(df = test_labels_df, 
                           data_directory = os.path.join(dataset_path, 'test'), 
                           transform = data_transf,
                           label_column = "nazi_cls_encoded")
    test_loader = DataLoader(dataset = test_data, batch_size = batch_size, shuffle=False)
    return train_loader, valid_loader, test_loader

In [9]:
batch_size = 16

train_loader, valid_loader, test_loader = create_datasets(batch_size, dataset_path)

In [11]:
device = get_device()
print(device)

train_dl = ToDeviceLoader(train_loader, device)
valid_dl = ToDeviceLoader(valid_loader, device)
test_dl = ToDeviceLoader(test_loader, device)

cuda


## Model Definition

In [12]:
def accuracy(predicted, actual):
    _, predictions = torch.max(predicted, dim=1)
    return torch.tensor(torch.sum(predictions==actual).item()/len(predictions))

In [13]:
class BaseModel(nn.Module):
    def training_step(self,batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out,labels)
        return loss
    
    def validation_step(self,batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out,labels)
        acc = accuracy(out,labels)
        return {"val_loss":loss.detach(),"val_acc":acc}
    
    def validation_epoch_end(self,outputs):
        batch_losses = [loss["val_loss"] for loss in outputs]
        loss = torch.stack(batch_losses).mean()
        batch_accuracy = [accuracy["val_acc"] for accuracy in outputs]
        acc = torch.stack(batch_accuracy).mean()
        return {"val_loss":loss.item(),"val_acc":acc.item()}
    
    def epoch_end(self, epoch, result):
        print("Epoch [{}], last_lr: {:.5f}, train_loss: {:.4f}, val_loss: {:.4f}, val_acc: {:.4f}".format(
            epoch, result['lrs'][-1], result['train_loss'], result['val_loss'], result['val_acc']))

In [14]:
def conv_shortcut(in_channel, out_channel, stride):
    layers = [nn.Conv2d(in_channel, out_channel, kernel_size=(1,1), stride=(stride, stride)),
             nn.BatchNorm2d(out_channel)]
    return nn.Sequential(*layers)

def block(in_channel, out_channel, k_size,stride, conv=False):
    layers = None
    
    first_layers = [nn.Conv2d(in_channel,out_channel[0], kernel_size=(1,1),stride=(1,1)),
                    nn.BatchNorm2d(out_channel[0]),
                    nn.ReLU(inplace=True)]
    if conv:
        first_layers[0].stride=(stride,stride)
    
    second_layers = [nn.Conv2d(out_channel[0], out_channel[1], kernel_size=(k_size, k_size), stride=(1,1), padding=1),
                    nn.BatchNorm2d(out_channel[1])]

    layers = first_layers + second_layers
    
    return nn.Sequential(*layers)
    

class ResNet(BaseModel):
    
    def __init__(self, in_channels, num_classes, pretrained=True):
        super().__init__()
        self.model = None
        if pretrained:
            self.model = timm.create_model('resnet34', in_chans=in_channels, num_classes=num_classes, pretrained=True)
        else:
        
            self.stg1 = nn.Sequential(
                                       nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=(3),
                                                 stride=(1), padding=1),
                                       nn.BatchNorm2d(64),
                                       nn.ReLU(inplace=True),
                                       nn.MaxPool2d(kernel_size=3, stride=2))
            
            ##stage 2
            self.convShortcut2 = conv_shortcut(64,256,1)
            
            self.conv2 = block(64,[64,256],3,1,conv=True)
            self.ident2 = block(256,[64,256],3,1)
    
            
            ##stage 3
            self.convShortcut3 = conv_shortcut(256,512,2)
            
            self.conv3 = block(256,[128,512],3,2,conv=True)
            self.ident3 = block(512,[128,512],3,2)
    
            
            ##stage 4
            self.convShortcut4 = conv_shortcut(512,1024,2)
            
            self.conv4 = block(512,[256,1024],3,2,conv=True)
            self.ident4 = block(1024,[256,1024],3,2)
            
            
            ##Classify
            self.classifier = nn.Sequential(
                                           nn.AvgPool2d(kernel_size=(4)),
                                           nn.Flatten(),
                                           nn.Linear(1024, num_classes))
        
    def forward(self,inputs):
        if self.model:
            return self.model(inputs)
        out = self.stg1(inputs)
        
        #stage 2
        out = F.relu(self.conv2(out) + self.convShortcut2(out))
        out = F.relu(self.ident2(out) + out)
        out = F.relu(self.ident2(out) + out)
        out = F.relu(self.ident2(out) + out)
        
        #stage3
        out = F.relu(self.conv3(out) + (self.convShortcut3(out)))
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        
        #stage4             
        out = F.relu(self.conv4(out) + (self.convShortcut4(out)))
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        
        #Classify
        out = self.classifier(out)#100x1024
        
        return out
        

In [15]:
model = ResNet(3,13, pretrained=True)

In [16]:
model = to_device(model, device)

In [17]:
@torch.no_grad()
def evaluate(model,valid_dl):
    model.eval()
    outputs = [model.validation_step(batch) for batch in valid_dl]
    return model.validation_epoch_end(outputs)

In [18]:

def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def fit (epochs, train_dl, valid_dl, model, optimizer, max_lr, weight_decay, scheduler, grad_clip=None, patience=10):
    torch.cuda.empty_cache()
    
    history = []
    
    optimizer = optimizer(model.parameters(), max_lr, weight_decay = weight_decay)
    
    scheduler = scheduler(optimizer, max_lr, epochs=epochs, steps_per_epoch=len(train_dl))

    early_stopping = EarlyStopping(patience=patience, verbose=True, path="resnet-output/resnet-multiple-pretrained-size-224-batch-16.pt")
    
    for epoch in range(epochs):
        model.train()
        
        train_loss = []
        
        lrs = []
        
        for batch in train_dl:
            loss = model.training_step(batch)
            
            train_loss.append(loss)
            
            loss.backward()
            
            if grad_clip:
                nn.utils.clip_grad_value_(model.parameters(), grad_clip)
            
            optimizer.step()
            optimizer.zero_grad()
            
            scheduler.step()
            lrs.append(get_lr(optimizer))
        result = evaluate(model, valid_dl)
        result["train_loss"] = torch.stack(train_loss).mean().item()
        result["lrs"] = lrs
        
        model.epoch_end(epoch,result)
        history.append(result)

        early_stopping(result["val_loss"], model)
        
        if early_stopping.early_stop:
            print("Early stopping")
            break
        
    return history

## Training the Model

In [19]:
epochs = 100
optimizer = torch.optim.Adam
max_lr = 1e-3
grad_clip = 0.1
weight_decay = 1e-5
scheduler = torch.optim.lr_scheduler.OneCycleLR
patience = 10

In [ ]:
%%time
history = fit(epochs=epochs, train_dl=train_dl, valid_dl=valid_dl, model=model, 
              optimizer=optimizer, max_lr=max_lr, grad_clip=grad_clip, patience=patience,
              weight_decay=weight_decay, scheduler=torch.optim.lr_scheduler.OneCycleLR)

Epoch [0], last_lr: 0.00004, train_loss: 1.7840, val_loss: 1.4058, val_acc: 0.5697
Validation loss decreased (inf --> 1.405808).  Saving model ...
Epoch [1], last_lr: 0.00005, train_loss: 1.1487, val_loss: 0.9410, val_acc: 0.7051
Validation loss decreased (1.405808 --> 0.940970).  Saving model ...
Epoch [3], last_lr: 0.00008, train_loss: 0.6207, val_loss: 0.5278, val_acc: 0.8276
Validation loss decreased (0.702260 --> 0.527784).  Saving model ...
Epoch [4], last_lr: 0.00010, train_loss: 0.4881, val_loss: 0.4484, val_acc: 0.8375
Validation loss decreased (0.527784 --> 0.448442).  Saving model ...
Epoch [5], last_lr: 0.00013, train_loss: 0.4007, val_loss: 0.3749, val_acc: 0.8641
Validation loss decreased (0.448442 --> 0.374913).  Saving model ...
Epoch [6], last_lr: 0.00016, train_loss: 0.3403, val_loss: 0.3378, val_acc: 0.8675
Validation loss decreased (0.374913 --> 0.337833).  Saving model ...
Epoch [7], last_lr: 0.00020, train_loss: 0.2851, val_loss: 0.3266, val_acc: 0.8670
Validation

In [29]:
with open('resnet-output/resnet-multi-history.json', 'w') as f:
    f.write(json.dumps(history))

## Evaluating the Model

In [26]:
model.load_state_dict(torch.load("resnet-output/resnet-multiple-pretrained-size-224-batch-16.pt", weights_only=True))
model.eval()

ResNet(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (drop_block): Identity()
        (act1): ReLU(inplace=True)
        (aa): Identity()
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act2): ReLU(inplace=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), 

In [27]:
# model.cpu()
outputs = []
y_true = []
for batch in test_dl:
    images, labels = batch
    out = model(images)
    _, predicted = torch.max(out, 1)
    # print([le.classes_[predicted[i]] for i in range(len(images))])
    y_true += labels.cpu()
    outputs += predicted.cpu()

## Metrics

In [28]:
print(classification_report(y_true, outputs, digits=3))
accuracy_score(y_true, outputs)

              precision    recall  f1-score   support

           0      0.926     0.806     0.862       124
           1      1.000     0.833     0.909        12
           2      0.909     0.625     0.741        16
           3      1.000     0.970     0.985        33
           4      0.842     0.924     0.881       185
           5      0.286     0.400     0.333         5
           6      1.000     1.000     1.000         3
           7      0.914     0.810     0.859       158
           8      0.765     0.946     0.846       261
           9      0.852     0.759     0.803       220
          10      1.000     0.500     0.667         8
          11      0.937     0.912     0.924       886
          12      0.622     0.848     0.718        33

    accuracy                          0.880      1944
   macro avg      0.850     0.795     0.810      1944
weighted avg      0.887     0.880     0.880      1944



0.8796296296296297

In [30]:
le.classes_

array(['black_sun', 'british_union_of_fascist', 'broken_sun_cross',
       'happy_merchant', 'hitler', 'hitler_salute', 'judenstern',
       'neo-nazi', 'siegrune', 'ss_skull', 'sturmabteilung_emblem',
       'swastika', 'wolfsangel'], dtype=object)

In [31]:
predicted_result = dict(y_true=y_true, outputs=outputs)
joblib.dump(predicted_result, "resnet-output/resnet-multi-predicted-result.joblib")

['resnet-multi-predicted-result.joblib']